# TMJDLC — Telegram Bot Runner

This notebook has one purpose: install/update the `drive` branch, mount Google Drive, load the two Telegram credentials, and run the bot while showing logs in Colab.

**Only two Colab Secrets are required:** `TG_BOT_TOKEN` and `TG_ADMIN_IDS`. All other settings use the defaults in the project.

The bot sends download/upload progress cards in Telegram. Runtime logs are shown below and also saved to `TMJDLC Library/logs/tmjdlc.log` on Google Drive.

## 1. Install or update TMJDLC

In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/TMJDLC')
if REPO_DIR.exists():
    %cd /content/TMJDLC
    !git fetch origin drive
    !git checkout drive
    !git pull --ff-only origin drive
else:
    %cd /content
    !git clone --branch drive --single-branch https://github.com/Telebotfaroff/TMJDLC.git
    %cd /content/TMJDLC

!pip -q install -r requirements.txt
!pip -q install -e .
!python -m playwright install --with-deps chromium
print('TMJDLC dependencies are ready.')

## 2. Mount Google Drive and prepare the persistent library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from tmjdlc.config import Settings
from tmjdlc.drive.manager import DriveManager
from tmjdlc.storage.json_store import JsonStore

# Project defaults are used for all settings except the two Telegram credentials.
settings = Settings.from_env() if 'TG_BOT_TOKEN' in __import__('os').environ else None
DRIVE_ROOT_NAME = 'TMJDLC Library'
LIBRARY_DIR = Path('/content/drive/MyDrive') / DRIVE_ROOT_NAME
LIBRARY_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = LIBRARY_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_PATH = LIBRARY_DIR / 'database.json'
WORK_DIR = Path('/content/TMJDLC-temp')
WORK_DIR.mkdir(parents=True, exist_ok=True)

database = JsonStore(DATABASE_PATH)
print('Google Drive library ready:', LIBRARY_DIR)
print('Persistent log file:', LOG_DIR / 'tmjdlc.log')

## 3. Load the only two required Colab Secrets

In Colab's Secrets panel, add `TG_BOT_TOKEN` and `TG_ADMIN_IDS`. Enable Notebook access for both.

In [ ]:
from google.colab import userdata
import os

os.environ['TG_BOT_TOKEN'] = userdata.get('TG_BOT_TOKEN').strip()
os.environ['TG_ADMIN_IDS'] = userdata.get('TG_ADMIN_IDS').strip()

if not os.environ['TG_BOT_TOKEN'] or not os.environ['TG_ADMIN_IDS']:
    raise RuntimeError('Add TG_BOT_TOKEN and TG_ADMIN_IDS in Colab Secrets and enable Notebook access.')

print('Both Telegram secrets loaded. Secret values are hidden.')

## 4. Start the bot — live logs appear in this output

Run this cell after the previous cells. Keep the Colab runtime alive. The log file is also written to Google Drive.

In [ ]:
import logging
import sys
from tmjdlc.config import Settings
from tmjdlc.drive.manager import DriveManager
from tmjdlc.bot.application import start_bot_async, stop_bot_async

settings = Settings.from_env()
settings.work_dir.mkdir(parents=True, exist_ok=True)

# Show logs in the Colab output and keep a persistent copy on Google Drive.
log_file = LOG_DIR / 'tmjdlc.log'
logging.basicConfig(
    level=getattr(logging, settings.log_level, logging.INFO),
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(log_file, encoding='utf-8')],
    force=True,
)
logging.getLogger('httpx').setLevel(logging.WARNING)

drive_manager = DriveManager.from_colab(root_name=settings.drive_root_name, root_id=settings.drive_root_id)
drive_root_id = drive_manager.ensure_root()
database = JsonStore(Path('/content/drive/MyDrive') / settings.drive_root_name / 'database.json')

bot_app = await start_bot_async(settings, drive_manager, database)
logging.getLogger('tmjdlc').info('TMJDLC bot started. Drive root ID: %s', drive_root_id)
print('Bot is running. Watch this cell output for logs; use the stop cell to shut it down.')

## Stop the bot

Run this cell to stop Telegram polling and shut down the workers cleanly.

In [ ]:
await stop_bot_async(bot_app)
logging.getLogger('tmjdlc').info('TMJDLC bot stopped cleanly.')